# 04) Train with W&B

## Setup

Loads `data/train.csv`/`data/columns.json` fresh (same convention as every other notebook here),
logs in to W&B (credentials already cached in `~/.netrc` -- no key ever passed here), and rebuilds
the exact 10-fold expanding-window CV split from `03)`: walk-forward, `MIN_TRAIN_DAYS=7` warm-up,
`DELTA_DAYS=1` step, no lookahead.

The loaded DataFrame is `train_df`, not `train` -- `train` is reserved for the per-model function
each W&B sweep calls.

In [1]:
import json
import os

import numpy as np
import pandas as pd
import wandb
from scipy.stats import spearmanr
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

train_df = pd.read_csv("data/train.csv", parse_dates=["date"])
with open("data/columns.json") as f:
    columns = json.load(f)

TARGET_COL = columns["target_cols"][0]
FINAL_FEATURE_COLS = columns["final_feature_cols"]

os.environ["WANDB_SILENT"] = "true"  # suppress wandb's own per-run console noise
wandb.login()

print(f"train_df: {train_df.shape[0]} rows, target = {TARGET_COL!r}")
print(f"{len(FINAL_FEATURE_COLS)} final features:", FINAL_FEATURE_COLS)

train_df: 8443 rows, target = 'target_bps_vs_mid'
4 final features: ['near_price_vs_mid_bps', 'urgency', 'ref_price_vs_mid_bps', 'far_price_vs_mid_bps']


In [2]:
unique_dates = np.sort(train_df["date"].unique())
MIN_TRAIN_DAYS = 7
DELTA_DAYS = 1

folds = []
for fold_idx, eval_start in enumerate(range(MIN_TRAIN_DAYS, len(unique_dates), DELTA_DAYS)):
    fit_dates = unique_dates[:eval_start]
    eval_dates = unique_dates[eval_start:eval_start + DELTA_DAYS]
    assert fit_dates.max() < eval_dates.min(), "eval dates must all fall after fit dates"

    fit_mask = train_df["date"].isin(fit_dates)
    eval_mask = train_df["date"].isin(eval_dates)
    folds.append((fit_mask, eval_mask))

    print(f"fold {fold_idx}: fit {fit_mask.sum():>4} rows ({len(fit_dates)} days, "
          f"{pd.Timestamp(fit_dates.min()).date()} to {pd.Timestamp(fit_dates.max()).date()}) | "
          f"eval {eval_mask.sum():>4} rows ({len(eval_dates)} days, "
          f"{pd.Timestamp(eval_dates.min()).date()} to {pd.Timestamp(eval_dates.max()).date()})")

fold 0: fit 3476 rows (7 days, 2025-03-03 to 2025-03-11) | eval  496 rows (1 days, 2025-03-12 to 2025-03-12)
fold 1: fit 3972 rows (8 days, 2025-03-03 to 2025-03-12) | eval  497 rows (1 days, 2025-03-13 to 2025-03-13)
fold 2: fit 4469 rows (9 days, 2025-03-03 to 2025-03-13) | eval  497 rows (1 days, 2025-03-14 to 2025-03-14)
fold 3: fit 4966 rows (10 days, 2025-03-03 to 2025-03-14) | eval  497 rows (1 days, 2025-03-17 to 2025-03-17)
fold 4: fit 5463 rows (11 days, 2025-03-03 to 2025-03-17) | eval  496 rows (1 days, 2025-03-18 to 2025-03-18)
fold 5: fit 5959 rows (12 days, 2025-03-03 to 2025-03-18) | eval  497 rows (1 days, 2025-03-19 to 2025-03-19)
fold 6: fit 6456 rows (13 days, 2025-03-03 to 2025-03-19) | eval  496 rows (1 days, 2025-03-20 to 2025-03-20)
fold 7: fit 6952 rows (14 days, 2025-03-03 to 2025-03-20) | eval  497 rows (1 days, 2025-03-21 to 2025-03-21)
fold 8: fit 7449 rows (15 days, 2025-03-03 to 2025-03-21) | eval  497 rows (1 days, 2025-03-24 to 2025-03-24)
fold 9: fit 7

## Shared sklearn sweep runner

One `train()`-function factory reused by every sklearn model's sweep (`HuberRegressor`, `Lasso`,
`GradientBoostingRegressor`, `RandomForestRegressor`): builds `impute -> scale -> model`, fits/evals
across all 10 folds, logs the full metric set. `random_state` is fixed to `0` automatically for
models that accept it (not swept -- a determinism knob, not a tunable choice).

In [3]:
def make_sklearn_train_fn(estimator_class):
    def train():
        wandb.init()
        config = dict(wandb.config)
        if "random_state" in estimator_class().get_params():
            config["random_state"] = 0
        estimator = estimator_class(**config)

        model = Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
            ("model", estimator),
        ])

        fit_mses, eval_mses = [], []
        fit_r2s, eval_r2s = [], []
        fit_ics, eval_ics = [], []
        for fit_mask, eval_mask in folds:
            X_fit = train_df.loc[fit_mask, FINAL_FEATURE_COLS]
            y_fit = train_df.loc[fit_mask, TARGET_COL]
            X_eval = train_df.loc[eval_mask, FINAL_FEATURE_COLS]
            y_eval = train_df.loc[eval_mask, TARGET_COL]

            model.fit(X_fit, y_fit)
            pred_fit = model.predict(X_fit)
            pred_eval = model.predict(X_eval)

            fit_mses.append(mean_squared_error(y_fit, pred_fit))
            eval_mses.append(mean_squared_error(y_eval, pred_eval))
            fit_r2s.append(r2_score(y_fit, pred_fit))
            eval_r2s.append(r2_score(y_eval, pred_eval))
            fit_ics.append(spearmanr(y_fit, pred_fit).statistic)
            eval_ics.append(spearmanr(y_eval, pred_eval).statistic)

        metrics = {
            "fit_mse": np.mean(fit_mses),
            "eval_mse_mean": np.mean(eval_mses),
            "eval_mse_std": np.std(eval_mses),
            "fit_r2": np.mean(fit_r2s),
            "eval_r2": np.mean(eval_r2s),
            "fit_ic": np.mean(fit_ics),
            "eval_ic": np.mean(eval_ics),
        }
        wandb.log(metrics)
        print(f"  [{wandb.run.name}] eval_ic={metrics['eval_ic']:.4f}  eval_mse={metrics['eval_mse_mean']:.2f}")
        wandb.finish()
    return train

print("make_sklearn_train_fn ready")

make_sklearn_train_fn ready


## HuberRegressor sweep (best linear)

`epsilon` controls where Huber's loss switches from quadratic to linear (robustness to outliers);
`alpha` is the L2 penalty strength. 30-trial Bayesian search, optimizing held-out IC.

In [4]:
from sklearn.linear_model import HuberRegressor

huber_sweep_config = {
    "method": "bayes",
    "metric": {"name": "eval_ic", "goal": "maximize"},
    "parameters": {
        "epsilon": {"distribution": "log_uniform_values", "min": 1.1, "max": 3.0},
        "alpha": {"distribution": "log_uniform_values", "min": 1e-5, "max": 1e1},
    },
}

huber_sweep_id = wandb.sweep(huber_sweep_config, project="close-auction-model-tuning")
wandb.agent(huber_sweep_id, function=make_sklearn_train_fn(HuberRegressor), count=30)
print("huber_sweep_id:", huber_sweep_id)

_api = wandb.Api()
_best = _api.sweep(f"{_api.default_entity}/close-auction-model-tuning/{huber_sweep_id}").best_run()
print(f"best: {_best.name}  eval_ic={_best.summary_metrics['eval_ic']:.4f}  "
      f"eval_mse_mean={_best.summary_metrics['eval_mse_mean']:.2f}  config={dict(_best.config)}")

Create sweep with ID: git37eu2
Sweep URL: https://wandb.ai/ablancoprada-massachusetts-institute-of-technology/close-auction-model-tuning/sweeps/git37eu2
  [dazzling-sweep-1] eval_ic=0.8206  eval_mse=9.8976
  [spring-sweep-2] eval_ic=0.8206  eval_mse=9.8709
  [jumping-sweep-3] eval_ic=0.8206  eval_mse=9.8695
  [smooth-sweep-4] eval_ic=0.8207  eval_mse=9.8337
  [serene-sweep-5] eval_ic=0.8206  eval_mse=9.8775
  [resilient-sweep-6] eval_ic=0.8207  eval_mse=10.4608
  [cool-sweep-7] eval_ic=0.8207  eval_mse=9.8337
  [fancy-sweep-8] eval_ic=0.8206  eval_mse=9.9152
  [clear-sweep-9] eval_ic=0.8206  eval_mse=9.8774
  [dainty-sweep-10] eval_ic=0.8206  eval_mse=9.9195
  [tough-sweep-11] eval_ic=0.8208  eval_mse=10.1806
  [worldly-sweep-12] eval_ic=0.8206  eval_mse=9.8819
  [deft-sweep-13] eval_ic=0.8203  eval_mse=9.9560
  [frosty-sweep-14] eval_ic=0.8206  eval_mse=9.9175
  [proud-sweep-15] eval_ic=0.8207  eval_mse=10.4608
  [generous-sweep-16] eval_ic=0.8207  eval_mse=9.8349
  [stellar-sweep-17]

## Lasso sweep (baseline)

Plain L1-penalized linear regression -- the explicit floor to compare the other four models
against. Only `alpha` to tune. 30-trial Bayesian search, optimizing held-out IC.

In [5]:
from sklearn.linear_model import Lasso

lasso_sweep_config = {
    "method": "bayes",
    "metric": {"name": "eval_ic", "goal": "maximize"},
    "parameters": {
        "alpha": {"distribution": "log_uniform_values", "min": 1e-4, "max": 1e1},
    },
}

lasso_sweep_id = wandb.sweep(lasso_sweep_config, project="close-auction-model-tuning")
wandb.agent(lasso_sweep_id, function=make_sklearn_train_fn(Lasso), count=30)
print("lasso_sweep_id:", lasso_sweep_id)

_api = wandb.Api()
_best = _api.sweep(f"{_api.default_entity}/close-auction-model-tuning/{lasso_sweep_id}").best_run()
print(f"best: {_best.name}  eval_ic={_best.summary_metrics['eval_ic']:.4f}  "
      f"eval_mse_mean={_best.summary_metrics['eval_mse_mean']:.2f}  config={dict(_best.config)}")

Create sweep with ID: m2dpwcxx
Sweep URL: https://wandb.ai/ablancoprada-massachusetts-institute-of-technology/close-auction-model-tuning/sweeps/m2dpwcxx
  [devoted-sweep-1] eval_ic=0.8208  eval_mse=10.0773
  [misunderstood-sweep-2] eval_ic=nan  eval_mse=72.0647
  [lively-sweep-3] eval_ic=0.8207  eval_mse=10.0785
  [comic-sweep-4] eval_ic=0.8207  eval_mse=10.0781
  [tough-sweep-5] eval_ic=0.8206  eval_mse=12.1086
  [autumn-sweep-6] eval_ic=0.8207  eval_mse=10.0776
  [serene-sweep-7] eval_ic=0.8207  eval_mse=10.0787
  [prime-sweep-8] eval_ic=0.8207  eval_mse=10.0773
  [toasty-sweep-9] eval_ic=0.8207  eval_mse=10.0777
  [fast-sweep-10] eval_ic=0.8207  eval_mse=10.0781
  [fine-sweep-11] eval_ic=0.8207  eval_mse=10.0781
  [fallen-sweep-12] eval_ic=0.8207  eval_mse=10.0773
  [mild-sweep-13] eval_ic=0.8208  eval_mse=10.0773
  [lemon-sweep-14] eval_ic=0.8207  eval_mse=10.0773
  [jumping-sweep-15] eval_ic=0.8207  eval_mse=10.0781
  [peachy-sweep-16] eval_ic=0.8207  eval_mse=10.0773
  [icy-sweep

## GradientBoostingRegressor sweep (best boosted tree)

`min_samples_leaf`'s range goes wide and regularization-leaning on purpose -- `03)`'s shortlist run
showed the untuned tree models overfitting badly, so the sweep needs room to find configs that
generalize, not just variations near sklearn's overfit-prone defaults. 30-trial Bayesian search,
optimizing held-out IC.

In [6]:
from sklearn.ensemble import GradientBoostingRegressor

gb_sweep_config = {
    "method": "bayes",
    "metric": {"name": "eval_ic", "goal": "maximize"},
    "parameters": {
        "n_estimators": {"distribution": "int_uniform", "min": 50, "max": 400},
        "max_depth": {"distribution": "int_uniform", "min": 2, "max": 6},
        "learning_rate": {"distribution": "log_uniform_values", "min": 1e-3, "max": 3e-1},
        "subsample": {"distribution": "uniform", "min": 0.5, "max": 1.0},
        "min_samples_leaf": {"distribution": "int_uniform", "min": 1, "max": 50},
    },
}

gb_sweep_id = wandb.sweep(gb_sweep_config, project="close-auction-model-tuning")
wandb.agent(gb_sweep_id, function=make_sklearn_train_fn(GradientBoostingRegressor), count=30)
print("gb_sweep_id:", gb_sweep_id)

_api = wandb.Api()
_best = _api.sweep(f"{_api.default_entity}/close-auction-model-tuning/{gb_sweep_id}").best_run()
print(f"best: {_best.name}  eval_ic={_best.summary_metrics['eval_ic']:.4f}  "
      f"eval_mse_mean={_best.summary_metrics['eval_mse_mean']:.2f}  config={dict(_best.config)}")

Create sweep with ID: 97pw7tca
Sweep URL: https://wandb.ai/ablancoprada-massachusetts-institute-of-technology/close-auction-model-tuning/sweeps/97pw7tca
  [feasible-sweep-1] eval_ic=0.8184  eval_mse=24.6609
  [wise-sweep-2] eval_ic=0.8180  eval_mse=24.2989
  [laced-sweep-3] eval_ic=0.8217  eval_mse=36.5363
  [distinctive-sweep-4] eval_ic=0.8201  eval_mse=47.5317
  [toasty-sweep-5] eval_ic=0.8219  eval_mse=30.7854
  [twilight-sweep-6] eval_ic=0.8219  eval_mse=40.1687
  [solar-sweep-7] eval_ic=0.8213  eval_mse=57.1514
  [polished-sweep-8] eval_ic=0.8212  eval_mse=50.5378
  [devout-sweep-9] eval_ic=0.8223  eval_mse=35.8114
  [quiet-sweep-10] eval_ic=0.8211  eval_mse=49.0043
  [charmed-sweep-11] eval_ic=0.8216  eval_mse=64.1094
  [chocolate-sweep-12] eval_ic=0.8220  eval_mse=36.9136
  [bright-sweep-13] eval_ic=0.8220  eval_mse=13.1347
  [spring-sweep-14] eval_ic=0.8218  eval_mse=23.4903
  [fancy-sweep-15] eval_ic=0.8228  eval_mse=12.2114
  [fast-sweep-16] eval_ic=0.8216  eval_mse=27.7187
 